# Lab 6 – Named Entity Recognition (NER)

## Task 1: Dataset Preparation

In [1]:
from datasets import load_dataset

dataset = load_dataset("wnut_17", trust_remote_code=True)
print(dataset)
print("\nFeatures:", dataset["train"].features["ner_tags"])

print(dataset["train"][0])


c:\repos\aait-lab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 3394
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1009
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1287
    })
})

Features: Sequence(feature=ClassLabel(names=['O', 'B-corporation', 'I-corporation', 'B-creative-work', 'I-creative-work', 'B-group', 'I-group', 'B-location', 'I-location', 'B-person', 'I-person', 'B-product', 'I-product'], id=None), length=-1, id=None)
{'id': '0', 'tokens': ['@paulwalk', 'It', "'s", 'the', 'view', 'from', 'where', 'I', "'m", 'living', 'for', 'two', 'weeks', '.', 'Empire', 'State', 'Building', '=', 'ESB', '.', 'Pretty', 'bad', 'storm', 'here', 'last', 'evening', '.'], 'ner_tags': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 8, 8, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0]}


In [2]:
label_list = dataset["train"].features["ner_tags"].feature.names

id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

print("id2label:", id2label)
print("label2id:", label2id)


id2label: {0: 'O', 1: 'B-corporation', 2: 'I-corporation', 3: 'B-creative-work', 4: 'I-creative-work', 5: 'B-group', 6: 'I-group', 7: 'B-location', 8: 'I-location', 9: 'B-person', 10: 'I-person', 11: 'B-product', 12: 'I-product'}
label2id: {'O': 0, 'B-corporation': 1, 'I-corporation': 2, 'B-creative-work': 3, 'I-creative-work': 4, 'B-group': 5, 'I-group': 6, 'B-location': 7, 'I-location': 8, 'B-person': 9, 'I-person': 10, 'B-product': 11, 'I-product': 12}


In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
print(tokenizer)




BertTokenizer(name_or_path='google-bert/bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})


In [4]:
def preprocess_function(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )
    
    labels_batch = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(labels[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels_batch.append(label_ids)
    
    tokenized_inputs["labels"] = labels_batch
    return tokenized_inputs

In [5]:
from evaluate import load
import numpy as np

seqeval = load("seqeval")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)
    true_predictions = []
    true_labels = []
    for prediction, label in zip(predictions, labels):
        true_predictions.append([id2label[p] for p, l in zip(prediction, label) if l != -100])
        true_labels.append([id2label[l] for p, l in zip(prediction, label) if l != -100])
                
    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return results


In [6]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
print("Data collator:", data_collator)

tokenized_dataset = dataset.map(preprocess_function, batched=True)
train_tokenized = tokenized_dataset["train"]
val_tokenized   = tokenized_dataset["validation"]

Data collator: DataCollatorForTokenClassification(tokenizer=BertTokenizer(name_or_path='google-bert/bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}), padding=True, max_length=None, pad_to_multiple_of=None, label_pad_token_id=-100, return_tensors='pt')


Map: 100%|██████████| 1009/1009 [00:00<00:00, 12007.73 examples/s]


In [7]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer


model = AutoModelForTokenClassification.from_pretrained(
    "google-bert/bert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)
print(model.config)

training_args = TrainingArguments(
    output_dir="bert-wnut-ner",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    
    # Hyperparameters (BERT paper recommendations)
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True,

    logging_steps=10,
    report_to="none",
)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 8754.89it/s]
BertForTokenClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not o

BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "O",
    "1": "B-corporation",
    "2": "I-corporation",
    "3": "B-creative-work",
    "4": "I-creative-work",
    "5": "B-group",
    "6": "I-group",
    "7": "B-location",
    "8": "I-location",
    "9": "B-person",
    "10": "I-person",
    "11": "B-product",
    "12": "I-product"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "B-corporation": 1,
    "B-creative-work": 3,
    "B-group": 5,
    "B-location": 7,
    "B-person": 9,
    "B-product": 11,
    "I-corporation": 2,
    "I-creative-work": 4,
    "I-group": 6,
    "I-location": 8,
    "I-person": 10,
    "

In [16]:
from transformers.utils.notebook import NotebookProgressCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

trainer.remove_callback(NotebookProgressCallback)
test_results = trainer.evaluate(eval_dataset=tokenized_dataset["test"])
print("Test results:", test_results)
print(f"F1: {test_results['eval_overall_f1']:.4f}")


Epoch,Training Loss,Validation Loss,Corporation,Creative-work,Group,Location,Person,Product,Overall Precision,Overall Recall,Overall F1,Overall Accuracy
1,0.028401,0.280729,"{'precision': 0.21739130434782608, 'recall': 0.29411764705882354, 'f1': 0.25, 'number': 34}","{'precision': 0.323943661971831, 'recall': 0.21904761904761905, 'f1': 0.26136363636363635, 'number': 105}","{'precision': 0.19101123595505617, 'recall': 0.4358974358974359, 'f1': 0.265625, 'number': 39}","{'precision': 0.6515151515151515, 'recall': 0.581081081081081, 'f1': 0.6142857142857142, 'number': 74}","{'precision': 0.7832957110609481, 'recall': 0.7382978723404255, 'f1': 0.7601314348302299, 'number': 470}","{'precision': 0.4166666666666667, 'recall': 0.2631578947368421, 'f1': 0.3225806451612903, 'number': 114}",0.597205,0.562201,0.579174,0.954098
2,0.010093,0.288554,"{'precision': 0.29411764705882354, 'recall': 0.29411764705882354, 'f1': 0.29411764705882354, 'number': 34}","{'precision': 0.3561643835616438, 'recall': 0.24761904761904763, 'f1': 0.29213483146067415, 'number': 105}","{'precision': 0.20454545454545456, 'recall': 0.23076923076923078, 'f1': 0.2168674698795181, 'number': 39}","{'precision': 0.5769230769230769, 'recall': 0.6081081081081081, 'f1': 0.5921052631578947, 'number': 74}","{'precision': 0.8129675810473815, 'recall': 0.6936170212765957, 'f1': 0.748564867967853, 'number': 470}","{'precision': 0.4155844155844156, 'recall': 0.2807017543859649, 'f1': 0.3350785340314136, 'number': 114}",0.633663,0.535885,0.580687,0.956005
3,0.010952,0.288863,"{'precision': 0.2571428571428571, 'recall': 0.2647058823529412, 'f1': 0.2608695652173913, 'number': 34}","{'precision': 0.38028169014084506, 'recall': 0.2571428571428571, 'f1': 0.3068181818181818, 'number': 105}","{'precision': 0.2, 'recall': 0.23076923076923078, 'f1': 0.21428571428571427, 'number': 39}","{'precision': 0.5972222222222222, 'recall': 0.581081081081081, 'f1': 0.5890410958904109, 'number': 74}","{'precision': 0.8110831234256927, 'recall': 0.6851063829787234, 'f1': 0.7427912341407151, 'number': 470}","{'precision': 0.3950617283950617, 'recall': 0.2807017543859649, 'f1': 0.3282051282051282, 'number': 114}",0.630528,0.528708,0.575146,0.955496


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.02it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

Test results: {'eval_loss': 0.34220582246780396, 'eval_corporation': {'precision': 0.2875, 'recall': 0.3484848484848485, 'f1': 0.3150684931506849, 'number': 66}, 'eval_creative-work': {'precision': 0.2909090909090909, 'recall': 0.22535211267605634, 'f1': 0.25396825396825395, 'number': 142}, 'eval_group': {'precision': 0.2934131736526946, 'recall': 0.296969696969697, 'f1': 0.29518072289156627, 'number': 165}, 'eval_location': {'precision': 0.541095890410959, 'recall': 0.5266666666666666, 'f1': 0.5337837837837839, 'number': 150}, 'eval_person': {'precision': 0.7608695652173914, 'recall': 0.5710955710955711, 'f1': 0.6524633821571237, 'number': 429}, 'eval_product': {'precision': 0.26666666666666666, 'recall': 0.15748031496062992, 'f1': 0.19801980198019803, 'number': 127}, 'eval_overall_precision': 0.49777777777777776, 'eval_overall_recall': 0.4151992585727525, 'eval_overall_f1': 0.45275391611925214, 'eval_overall_accuracy': 0.9472446667521697, 'eval_runtime': 2.0241, 'eval_samples_per_sec

In [25]:
from transformers import pipeline
import spacy
from spacy.tokens import Doc, Span
from spacy import displacy

#default
text = ("The Golden State Warriors are an American professional basketball team based in San Francisco.")

# person + location
#text = "Elon Musk arrived in Tokyo for a Tesla shareholders meeting."

# creative-work + person  
#text = "Just watched Oppenheimer with Christopher Nolan, absolute masterpiece."

# group + product
#text = "Apple announced the iPhone 16 Pro at their Worldwide Developers Conference."

# corporation + location
#text = "Microsoft is opening a new campus in Warsaw, Poland next year."

# noisy Twitter-style (closer to WNUT-17 training data)
#text = "omg saw Taylor Swift at coachella she was performing w/ Bad Bunny 🔥🔥"

# multiple entity types
#text = "The LA Lakers beat the Boston Celtics 112-98 at the Crypto.com Arena last night."

model_path = "C:\\repos\\aait-lab\\bert-wnut-ner\\checkpoint-639"

classifier = pipeline("ner", model=model_path,
                        aggregation_strategy="simple")
entities = classifier(text)

nlp = spacy.blank("en")
doc = nlp.make_doc(text)
spans = []

for ent in entities:
    span = doc.char_span(ent["start"], ent["end"],
                        label=ent["entity_group"])
    if span is not None:
        spans.append(span)
doc.ents = spans

displacy.render(doc, style="ent", jupyter=True) # for jupyter
# displacy.serve(doc, style="ent") # to run the local server

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6164.95it/s]
